In [1]:
import tensorflow as tf 
from tensorflow import keras
import numpy as np 

from sklearn.metrics import mean_absolute_error
import tensorflow_addons as tfa

c:\Users\SIA\anaconda3\envs\deepface-env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Load data

In [2]:
train_ds = tf.data.experimental.load('./data/audio/train_ds/') \
    .cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)


valid_ds = tf.data.experimental.load('./data/audio/val_ds/') \
    .cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


In [3]:
train_ds, valid_ds

(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Build Audio model

In [4]:
def make_audio_model():
  inputs = keras.layers.Input(shape=(15,128))

  x = keras.layers.Conv1D(32, 2)(inputs)
  x = keras.layers.Dropout(0.3)(x)
  
  x = keras.layers.Conv1D(64, 2)(x)
  x = keras.layers.Dropout(0.3)(x)

  x = keras.layers.LSTM(512, return_sequences=True)(x)
  x = keras.layers.LSTM(256)(x)

  x = keras.layers.Dense(256)(x)
  x = keras.layers.Dropout(0.3)(x)


  x = keras.layers.Dense(5, activation='sigmoid')(x)

  return keras.models.Model(inputs=inputs, outputs=x, name='audio_model')


audio_model = make_audio_model()


### Compile model

In [ ]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

# optimizer = keras.optimizers.Adam(learning_rate=0.001)
# optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
optimizer = tfa.optimizers.RectifiedAdam(learning_rate=0.001)

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/audio/'+str(t)+'/audio.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

audio_model.compile(loss='mse', optimizer=optimizer , metrics=['mae'])

### Train

In [6]:
history = audio_model.fit(train_ds, validation_data=valid_ds, batch_size=32, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
1/1 [==============================] - 5s 5s/step - loss: 0.0266 - mae: 0.1474 - val_loss: 0.0120 - val_mae: 0.0947
Epoch 2/100
1/1 [==============================] - 0s 264ms/step - loss: 0.0271 - mae: 0.1501 - val_loss: 0.0120 - val_mae: 0.0946
Epoch 3/100
1/1 [==============================] - 0s 256ms/step - loss: 0.0294 - mae: 0.1567 - val_loss: 0.0119 - val_mae: 0.0944
Epoch 4/100
1/1 [==============================] - 0s 266ms/step - loss: 0.0297 - mae: 0.1569 - val_loss: 0.0119 - val_mae: 0.0943
Epoch 5/100
1/1 [==============================] - 0s 300ms/step - loss: 0.0315 - mae: 0.1633 - val_loss: 0.0119 - val_mae: 0.0941
Epoch 6/100
1/1 [==============================] - 0s 257ms/step - loss: 0.0270 - mae: 0.1530 - val_loss: 0.0105 - val_mae: 0.0871
Epoch 7/100
1/1 [==============================] - 0s 261ms/step - loss: 0.0290 - mae: 0.1586 - val_loss: 0.0089 - val_mae: 0.0783
Epoch 8/100
1/1 [==============================] - 0s 257ms/step - loss: 0.0209 - mae:

### Load weights

In [7]:
audio_model.load_weights('./weights/audio/audio.t5')

## Evaluation

### Validation data

In [8]:
train_ds = tf.data.experimental.load('./data/audio/train_ds')

y_true = np.concatenate([y for x,y in train_ds])
y_pred = audio_model.predict(train_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 1s 869ms/step


(array([90.79926 , 88.22617 , 94.586525, 90.76921 , 90.30091 ],
       dtype=float32),
 90.93641862273216)

### Validation data

In [9]:
valid_ds = tf.data.experimental.load('./data/audio/val_ds')

y_true = np.concatenate([y for x,y in valid_ds])
y_pred = audio_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 34ms/step


(array([92.73004 , 97.894936, 99.57511 , 96.56365 , 94.404526],
       dtype=float32),
 96.23365290462971)

### Test data

In [10]:
test_ds = tf.data.experimental.load('./data/audio/test_ds/')
loss, mae = audio_model.evaluate(test_ds)


y_true = np.concatenate([y for x,y in test_ds])
y_pred = audio_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 0s 40ms/step


(array([88.21736 , 85.4573  , 96.3691  , 99.57762 , 96.600555],
       dtype=float32),
 93.24438720941544)